## Package Import

In [1]:
!pip install --quiet h5py einops jiwer
#!pip install --quiet https://github.com/kpu/kenlm/archive/master.zip pyctcdecode

import os
import sys,random
import re
import math
import shutil
import subprocess
import collections
import multiprocessing
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
from tqdm.auto import tqdm
import jiwer
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW

# --- 3. GLOBAL CONFIGURATION ---
print("Torch", torch.__version__, "CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOGIT_TO_PHONEME = [
'BLANK',    # "BLANK" = CTC blank symbol
'AA', 'AE', 'AH', 'AO', 'AW',
'AY', 'B', 'CH', 'D', 'DH',
'EH', 'ER', 'EY', 'F', 'G',
'HH', 'IH', 'IY', 'JH', 'K',
'L', 'M', 'N', 'NG', 'OW',
'OY', 'P', 'R', 'S', 'SH',
'T', 'TH', 'UH', 'UW', 'V',
'W', 'Y', 'Z', 'ZH',
'|',    # "|" = silence token
]


Torch 2.2.0+cu121 CUDA available: True


### Hyperparameters

In [2]:
downsample_factor = 6 #downsample timeseries by a factor
train = True
test=True

## Signal2Phoneme

### Data Loading

In [3]:
dates_dict = {'2023.08.11': 0, '2023.08.13': 1, '2023.08.18': 2, '2023.08.20': 3, '2023.08.25': 4, '2023.08.27': 5, '2023.09.01': 6, '2023.09.03': 7, '2023.09.24': 8, '2023.09.29': 9, '2023.10.01': 10, '2023.10.06': 11, '2023.10.08': 12, '2023.10.13': 13, '2023.10.15': 14, '2023.10.20': 15, '2023.10.22': 16, '2023.11.03': 17, '2023.11.04': 18, '2023.11.17': 19, '2023.11.19': 20, '2023.11.26': 21, '2023.12.03': 22, '2023.12.08': 23, '2023.12.10': 24, '2023.12.17': 25, '2023.12.29': 26, '2024.02.25': 27, '2024.03.03': 28, '2024.03.08': 29, '2024.03.15': 30, '2024.03.17': 31, '2024.04.25': 32, '2024.04.28': 33, '2024.05.10': 34, '2024.06.14': 35, '2024.07.19': 36, '2024.07.21': 37, '2024.07.28': 38, '2025.01.10': 39, '2025.01.12': 40, '2025.03.14': 41, '2025.03.16': 42, '2025.03.30': 43, '2025.04.13': 44}

In [4]:
def _discover_train_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_train.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _discover_val_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_val.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _sorted_trials(hf):
    trials = [k for k in hf.keys() if k.startswith("trial_")]
    def idx(k):
        m = re.search(r'trial_(\d+)', k)
        return int(m.group(1)) if m else 0
    return sorted(trials, key=idx)

def _parse_session_date(session_name: str):
    # session folder like 't15.2023.08.13' -> (2023, 8, 13)
    m = re.search(r'(\d{4})\.(\d{2})\.(\d{2})', session_name)
    return tuple(map(int, m.groups())) if m else (9999, 99, 99)

INPUT_DIR = Path('./t15_copyTask_neuralData/hdf5_data_final')
train_files = _discover_train_files_in_order(INPUT_DIR)
val_files = _discover_val_files_in_order(INPUT_DIR)

class BrainDataset(Dataset):
    def __init__(self):
        self.all_feats = []
        self.all_tgt = []
        self.dates = []
        self.all_rate = []
        self.all_slength = []
        self.all_tlength = []
        self.all_targets = []
        self.load_data_to_RAM()
        
    def __len__(self):
        return len(self.all_feats)

    def _open_file(self, path):
        if path in self._file_cache:
            self._file_cache.move_to_end(path)
            return self._file_cache[path]
        f = h5py.File(path, 'r')
        self._file_cache[path] = f
        if len(self._file_cache) > self._cache_size:
            old_path, old_f = self._file_cache.popitem(last=False)
            try: old_f.close()
            except: pass
        return f
    
    def load_data_to_RAM(self):
        for train_path in tqdm(train_files):
            with h5py.File(train_path, "r") as hf:
                match = re.search(r"\d{4}\.\d{2}\.\d{2}", str(train_path))
                date = torch.tensor([dates_dict[match.group(0)]]).to(device)
                for trial_name in _sorted_trials(hf):
                    self.all_feats.append(torch.from_numpy(hf[trial_name]["input_features"][()].astype("float32")))
                    self.all_tgt.append(torch.from_numpy(hf[trial_name]['seq_class_ids'][()].astype('int64')))
                    self.dates.append(date)
        
        for train_path in tqdm(val_files):
            with h5py.File(train_path, "r") as hf:
                match = re.search(r"\d{4}\.\d{2}\.\d{2}", str(train_path))
                date = torch.tensor([dates_dict[match.group(0)]]).to(device)
                for trial_name in _sorted_trials(hf):
                    self.all_feats.append(torch.from_numpy(hf[trial_name]["input_features"][()].astype("float32")))
                    self.all_tgt.append(torch.from_numpy(hf[trial_name]['seq_class_ids'][()].astype('int64')))
                    self.dates.append(date)
        
    def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         f = self._open_file(row['h5_path'])
#         g = f[row['group']]
#         feats = g['input_features'][()].astype('float32')
#         tgt = g['seq_class_ids'][()].astype('int64')
#         date = row['date']
        return self.all_feats[idx], self.all_tgt[idx], self.dates[idx]
    
    def check_len_x_len_y_rate(self):
        return self.all_rate


    def collate_for_ctc(self,batch, pad_id=0, blank_id=0, enforce_multiple=6):
        """
        batch: list of (x, y)
          x: Tensor [T_x, C]  (variable T_x)
          y: 1D LongTensor [T_y] possibly padded with pad_id (e.g., 0)
        pad_id: the value used in dataset to pad target sequences
        blank_id: CTC blank index (must NOT appear in cleaned targets)
        enforce_multiple: pad time T_x up to multiple of this (e.g., for strided convs)

        Returns:
          x_padded: [B, C, T_max]  (you can permute later if needed)
          targets_concat: 1D LongTensor of all cleaned targets concatenated
          x_lens: [B] input lengths before time padding
          t_lens: [B] cleaned target lengths (no pads, no blanks)
        """
        xs, ys,dates = zip(*batch)

        # --- input time lengths and padding to multiple ---
        x_lens = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
        T_max = int(x_lens.max().item())
        if enforce_multiple is not None and enforce_multiple > 1:
            mod = T_max % enforce_multiple
            if mod != 0:
                T_max += (enforce_multiple - mod)

        C = xs[0].shape[1]
        B = len(xs)
        x_padded = torch.zeros(B, C, T_max, dtype=torch.float32)
        for i, x in enumerate(xs):
            # x is [T, C] -> store as [B, C, T]
            x_padded[i, :, :x.shape[0]] = x.permute(1, 0)

        # --- clean targets: remove pads, ensure no blank in targets ---
        ys = torch.stack(ys)  # (B, T_max)
        mask = ys != pad_id
        t_lens = mask.sum(dim=1)  # (B,)
        targets_concat = ys[mask]
#         self.all_slength.append(x_lens)
#         self.all_tlength.append(t_lens)
#         self.all_rate.append(x_lens/t_lens)
#         self.all_targets.append(targets_concat)
        
        # final sanity checks
        if targets_concat.numel() > 0:
            assert (targets_concat != blank_id).all(), "Targets still contain blank id after cleaning."
        return x_padded, targets_concat, x_lens, t_lens, dates


In [5]:
if train or test: train_ds = BrainDataset()

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/41 [00:00<?, ?it/s]

In [6]:
np.random.seed(10)
all_idx = len(train_ds)
all_index = np.random.permutation(all_idx)

In [7]:
if train or test:
    train_dataset = torch.utils.data.Subset(train_ds, all_index[:all_idx//10*9+800])
    test_dataset  = torch.utils.data.Subset(train_ds, all_index[all_idx//10*9+800:])
#     real_val = torch.utils.data.Subset(train_ds, list(range(8071,8071+1425)))
    # Set num_workers=0 to disable multiprocessing for stability in Kaggle.
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_ds.collate_for_ctc, num_workers=0)
    val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=train_ds.collate_for_ctc, num_workers=0)
#     real_val_loader = DataLoader(real_val, batch_size=64, shuffle=False, collate_fn=train_ds.collate_for_ctc, num_workers=0)

    print(len(train_loader), len(val_loader))
    print(f"\nDataLoaders created. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

292 5

DataLoaders created. Train batches: 292, Val batches: 5


In [8]:
def seed_everything(seed: int = 42):
    # Python built in random
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make CuDNN deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optional but recommended
    os.environ["PYTHONHASHSEED"] = str(seed)

# usage
seed_everything(42)

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(1000) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x, train=False): 
        if train:
            shift = random.randint(0,10)
            return x + self.pe[:, shift:x.size(1)+shift]
        else:
            return x + self.pe[:, :x.size(1)]

class ConvStem(nn.Module):
    def __init__(self, in_ch, d_model):
        super().__init__()
        self.d_model = d_model
        self.in_ch = in_ch

        self.cov0 = nn.Sequential(nn.Conv1d(in_ch, d_model , kernel_size=17, stride=3, padding=8,bias=False),nn.ReLU())
        self.cov01 = nn.Sequential(nn.Conv1d(d_model , d_model, kernel_size=11, stride=2, padding=5,bias=False),nn.ReLU(),
                                  nn.Conv1d(d_model , d_model//2, kernel_size=7, stride=1, padding=3,bias=False),nn.ReLU(),)
        
        self.cov1 = nn.Sequential(nn.Conv1d(in_ch, d_model , kernel_size=11, stride=3, padding=5,bias=False),nn.ReLU())
        self.cov11 = nn.Sequential(nn.Conv1d(d_model, d_model, kernel_size=7, stride=2, padding=3,bias=False),nn.ReLU(),
                                  nn.Conv1d(d_model, d_model//2, kernel_size=5, stride=1, padding=2,bias=False),nn.ReLU(),)
        
        self.cov2 = nn.Sequential(nn.Conv1d(in_ch, d_model, kernel_size=5, stride=3, padding=2,bias=False),nn.ReLU())
        self.cov21 = nn.Sequential(nn.Conv1d(d_model, d_model//2 , kernel_size=3, stride=2, padding=1,bias=False),nn.ReLU(),)

        
    def forward(self, x): 
        emb1 = self.cov01(self.cov0(x)).permute(0, 2, 1)
        emb2 = self.cov11(self.cov1(x)).permute(0, 2, 1)
        emb3 = self.cov21(self.cov2(x)).permute(0, 2, 1)
        return emb1,emb2,emb3

class BrainToTextModel(nn.Module):
    def __init__(self, in_ch=512//2, d_model=512, nhead=4, num_layers=2, vocab_size=len(LOGIT_TO_PHONEME)):
        super().__init__()
        print(vocab_size,'pheneme used')
        scaler = 4
        day_dim = 32
        self.conv = ConvStem(in_ch+day_dim, d_model)
        self.pos_enc = PositionalEncoding(scaler*d_model//8)
        
        encoder_layer1 = nn.TransformerEncoderLayer(
            d_model=scaler*d_model//8, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, activation='relu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer1, num_layers=num_layers)

        self.fc = nn.Sequential(nn.Linear(scaler*d_model//8, vocab_size,bias=False))
        self.act=nn.ReLU()
        
        self.day_reminder = nn.Parameter(nn.init.xavier_uniform_(torch.empty(len(dates_dict),day_dim)))
        
    def forward(self, x, day_idx, train=False):
        day_reminder = self.day_reminder[day_idx][:,:,None].repeat(1,1,x.shape[-1])
        mask = x!=0
        x = torch.cat([x,day_reminder], 1)
        x *= mask[:,0:1,:]
        
        emb1,emb2,emb3 = self.conv(x)
        x = 0.5*emb1+0.3*emb2+0.2*emb3
        x = self.pos_enc(x,train)
        x = self.act(self.transformer(x))
        logits = self.fc(x)
        return F.log_softmax(logits, dim=2)


model = BrainToTextModel().to(device)
print(f"Model Initialized. Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} Million")

41 pheneme used
Model Initialized. Total parameters: 14.19 Million


e:\python\lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [10]:
import random
def batch_time_stretch_to_factor(
    x: torch.Tensor,
    min_rate: float = 0.80,
    max_rate: float = 1.25,
):
    """
    x: [B, C, T] tensor
    Returns:
        x_out: [B, C, T_pad] where T_pad is the smallest multiple of
               downsample_factor that is >= round(T * rate)
        rate:  the stretch rate used
    """
    B, C, T = x.shape

    # One stretch rate for entire batch
    rate = random.uniform(min_rate, max_rate)

    # New length after stretching
    new_len = max(1, int(round(T * rate)))

    # Stretch whole batch in one shot
    x_stretch = F.interpolate(
        x, size=new_len, mode="linear", align_corners=False
    )  # [B, C, new_len]

    # Compute padded length as multiple of downsample_factor
    if downsample_factor <= 0:
        raise ValueError("downsample_factor must be positive")

    T_pad = math.ceil(new_len / downsample_factor) * downsample_factor

    # If already a multiple, no padding
    if T_pad == new_len:
        return x_stretch, rate

    pad_right = T_pad - new_len
#     Pad along time dimension on the right: (left, right)
    x_out = F.pad(x_stretch, (0, pad_right))

    return x_out, rate


def indices_to_phonemes(tensor_indices, remove_blank=True):
    # Convert tensor to list if needed
    if isinstance(tensor_indices, torch.Tensor):
        tensor_indices = tensor_indices.tolist()
    
    phonemes = [LOGIT_TO_PHONEME[i] for i in tensor_indices]
    
    if remove_blank:
        phonemes = [p for p in phonemes if p not in ['BLANK',]]
    return phonemes

In [11]:
from scripts.check_confusion import build_phoneme_confusion_from_str_indices
from scripts import data_augmentation

import importlib
importlib.reload(data_augmentation)
gauss_smooth, TimeSeriesAugment = data_augmentation.gauss_smooth, data_augmentation.TimeSeriesAugment

NUM_EPOCHS = 200
MAX_LR = 2e-3
WEIGHT_DECAY = 1e-5
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
best_wer = float('inf')

augment = TimeSeriesAugment()

import matplotlib.pyplot as plt
use_trained_model = True
if not use_trained_model:
    print(f"🚀 Starting training for {NUM_EPOCHS} epochs with OneCycleLR scheduler.")
    for epoch in range(1, NUM_EPOCHS + 1):
        optimizer = Adam(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [train]", leave=False)
        deleted_frames, total_loss, sample_count, total_val_loss = 0,0,0,0
        all_refs,all_preds = [], []
        for i, (x, targets, x_lens, t_lens, dates) in enumerate(pbar):
            if targets.numel() == 0: continue
            x, targets, x_lens, t_lens, dates = x.to(device), targets.to(device), x_lens.to(device), t_lens.to(device), torch.tensor(list(dates)).to(device)
            factor = 1
            pl = 0
            if random.random()<0.90:
                x, pl = augment(x)
                if random.random()<0.8:
                    x, factor = batch_time_stretch_to_factor(x)
            x = x[:,256:,:]
            x = gauss_smooth(x.transpose(2,1),device)
            input_lengths = torch.ceil((x_lens+pl)*factor).long()//downsample_factor
            log_probs = model(x,dates,True)
            loss = ctc_loss(log_probs.permute(1, 0, 2), targets, input_lengths, t_lens)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss+=loss.item()
            pbar.set_postfix({'loss': loss.item(), 'lr': MAX_LR})#scheduler.get_last_lr()[0]})
            
        if epoch<75 and epoch>5:
            MAX_LR/=1.04
        elif epoch<175 and epoch>5:
            MAX_LR/=1.02
        model.eval()
        with torch.no_grad():
            for x, targets, x_lens, t_lens, dates in tqdm(val_loader, desc=f"Epoch {epoch} [val]", leave=False):
                if targets.numel() == 0: continue
                x, targets, x_lens, t_lens, dates = x.to(device), targets.to(device), x_lens.to(device), t_lens.to(device), torch.tensor(list(dates)).to(device)
                x = x[:,256:,:]
                x = gauss_smooth(x.transpose(2,1),device)
                
                log_probs = model(x,dates) 
                input_lengths = x_lens//downsample_factor

                total_val_loss += ctc_loss(log_probs.permute(1, 0, 2), targets, input_lengths, t_lens).item()

                decoded = log_probs.argmax(-1).cpu().numpy()
                target_offset = 0
                for i, L in enumerate(t_lens.cpu().numpy()):
                    pred_indices = [p for j,p in enumerate(decoded[i][:input_lengths[i]]) if (j==0 or p!=decoded[i][j-1]) and p!=0]
                    all_preds.append(" ".join(map(str, pred_indices)))
                    all_refs.append(" ".join(map(str, targets[target_offset:target_offset+L].cpu().numpy())))
                    target_offset += L
        
        avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0
        
        
        wer = jiwer.wer(all_refs, all_preds) if len(all_refs) > 0 else 1.0
        print(f"Epoch {epoch}: Train Loss={total_loss/len(train_loader):.4f} | Val Loss={avg_val_loss:.4f} | WER={wer:.4f}")
        if wer < best_wer:
            best_wer = wer
            build_phoneme_confusion_from_str_indices(all_refs, all_preds)
            torch.save(model.state_dict(), f"./model_saving/best_model.pth")
            print(f"✅ New best model saved with WER: {best_wer:.4f}") 
else:
    state_dict = torch.load(
    r"F:\Kaggle\Brain2text\model_saving\best_model.pth",
    map_location="cpu"
    )
 
    model.load_state_dict(state_dict)
    model.eval()
    print("model loaded!")

model loaded!


In [12]:
# all_rate = torch.cat(train_ds.all_rate).flatten().cpu().numpy()
# all_slength = torch.cat(train_ds.all_slength).flatten()
# all_tlength = torch.cat(train_ds.all_tlength).flatten()

# # 2. Plot — e.g., scatter plots
# plt.figure(figsize=(12, 5))

# plt.subplot(1, 2, 1)
# plt.scatter(all_rate, all_slength, alpha=0.5)
# plt.xlabel("Rate")
# plt.ylabel("Source Length")
# plt.title("Rate vs Source Length")

# plt.subplot(1, 2, 2)
# plt.scatter(all_rate, all_tlength, alpha=0.5, color="orange")
# plt.xlabel("Rate")
# plt.ylabel("Target Length")
# plt.title("Rate vs Target Length")

# plt.tight_layout()
# plt.show()

# all_targets_t = torch.cat(train_ds.all_targets).cpu().flatten().numpy()
# all_targets_v = torch.cat(val_ds.all_targets).cpu().flatten().numpy()

# num_classes = len(LOGIT_TO_PHONEME)
# counts = np.bincount(all_targets_t, minlength=num_classes)
# print(np.unique(all_targets_t))

# # Optional: exclude special tokens (BLANK=0, <pad>=1, SIL=-1)
# mask = np.ones(num_classes, dtype=bool)
# mask[[0, 1]] = False  # exclude BLANK, <pad>, and SIL
# # counts = counts[mask]
# labels = np.array(LOGIT_TO_PHONEME)#[mask]
# # Plot
# plt.figure(figsize=(12, 5))
# plt.bar(np.arange(len(labels)), counts)
# plt.xticks(np.arange(len(labels)), labels, rotation=90)
# plt.xlabel("Phoneme")
# plt.ylabel("Frequency")
# plt.title("Phoneme Frequency Distribution")
# plt.tight_layout()
# plt.show()
# print(counts/counts.sum())


# counts = np.bincount(all_targets_v, minlength=num_classes)
# print(np.unique(all_targets_v))
# # Optional: exclude special tokens (BLANK=0, <pad>=1, SIL=-1)
# mask = np.ones(num_classes, dtype=bool)
# mask[[0, 1]] = False  # exclude BLANK, <pad>, and SIL
# counts = counts[mask]
# labels = np.array(LOGIT_TO_PHONEME)#[mask]

# # Plot
# plt.figure(figsize=(12, 5))
# plt.bar(np.arange(len(labels)), counts)
# plt.xticks(np.arange(len(labels)), labels, rotation=90)
# plt.xlabel("Phoneme")
# plt.ylabel("Frequency")
# plt.title("Phoneme Frequency Distribution")
# plt.tight_layout()
# plt.show()
# print(counts/counts.sum())


### Finetune LLM

#### Data Preparation

In [13]:
INPUT_DIR = Path('./t15_copyTask_neuralData/hdf5_data_final')
def codes_to_sentence(codes, encoding="utf-8"):
    """
    Accepts list, numpy array, or torch tensor of int codes.
    Removes zeros used as padding, then decodes to text.
    """
    try:
        import torch
        if isinstance(codes, torch.Tensor):
            codes = codes.detach().cpu().numpy()
    except Exception:
        pass

    arr = np.asarray(codes, dtype=np.uint8).ravel()
    arr = arr[arr != 0]              # drop padding zeros anywhere
    return bytes(arr).decode(encoding, errors="ignore")


In [14]:
import json
import h5py
import numpy as np
from tqdm import tqdm
from glob import glob
import os

def _discover_train_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_train.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    for p in input_dir.rglob("data_val.hdf5"):
        session = p.parent.name
        items.append((p, session))
#     items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _sorted_trials(hf):
    trials = [k for k in hf.keys() if k.startswith("trial_")]
    def idx(k):
        m = re.search(r'trial_(\d+)', k)
        return int(m.group(1)) if m else 0
    return sorted(trials, key=idx)

def indices_to_phonemes(tensor_indices, remove_blank=True):
    # Convert tensor to list if needed
    if isinstance(tensor_indices, torch.Tensor):
        tensor_indices = tensor_indices.tolist()
    
    phonemes = [LOGIT_TO_PHONEME[i] for i in tensor_indices]
    
    if remove_blank:
        phonemes = [p for p in phonemes if p not in ['BLANK',]]
    return phonemes


train_files = _discover_train_files_in_order(INPUT_DIR)
output_jsonl = "./data/phoneme_train.jsonl"
num_written = 0
with open(output_jsonl, "w", encoding="utf-8") as fout:
    for train_path in tqdm(train_files, desc="Inference with LM (chronological)"):
        with h5py.File(train_path, "r") as hf:
            for trial_name in _sorted_trials(hf):
                transcript = hf[trial_name]["transcription"][()]
                sentence = codes_to_sentence(transcript)
                phoneme = hf[trial_name]['seq_class_ids'][()].astype('int64')
                phoneme = phoneme[phoneme!=0]
                phoneme = " ".join(indices_to_phonemes(phoneme))
                record = {
                            "phonemes": phoneme,
                            "target": sentence,
                        }
                fout.write(json.dumps(record, ensure_ascii=False) + "\n")
                num_written += 1
print(f"Done. Wrote {num_written} samples → {output_jsonl}")

Inference with LM (chronological): 100%|███████████████████████████████████████████████| 86/86 [00:05<00:00, 16.48it/s]

Done. Wrote 9498 samples → ./data/phoneme_train.jsonl


#### Train

In [15]:
import random
import re

LABELS = [
    'BLANK',    # 0
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B', 'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH',
    '|',      # silence token
    '<DEL>',    # pseudo phoneme for deletion
    '<INS>',    # pseudo phoneme for insertion
]

# phonemes we will use for switch and insert, excluding BLANK and separator

import numpy as np

SPECIAL_TOKENS = ['BLANK', '|', '<INS>']

def build_replacement_table(conf_df, labels):
    """
    conf_df: DataFrame, index=labels, columns=labels (counts)
    labels: list of phoneme labels (same order as conf_df)
    
    Returns:
        replacement_table: dict[ref_ph] = (targets, probs)
    """
    replacement_table = {}
    total_correct, counter, err_count = 0, 0, 0
    for ref_ph in labels:
        if ref_ph in SPECIAL_TOKENS:
            continue  # we typically do not noise these
        row = conf_df.loc[ref_ph]  # Series: counts of predicted phonemes given ref_ph
        # Exclude special tokens and self as switch targets
        valid_targets = [
            ph for ph in labels
            if ph not in SPECIAL_TOKENS and ph != ref_ph
        ]

        if not valid_targets:
            continue

        counts = np.array([row.get(t, 0) for t in valid_targets], dtype=float)
#         counts = counts-1
#         counts[counts<0] = 0
        corrects = np.array([row.get(ref_ph, 0)])
        total_correct += corrects
        err_count += counts.sum()
        counter += counts.sum()+corrects
        # If all counts are zero, fall back to uniform
        if counts.sum() == 0:
            probs = np.ones_like(counts) * 0
        else:
            # Normalize counts per row to [0,1]
            counts_sum_all = counts.sum()+corrects
            counts_all = np.concatenate([counts,corrects])
            valid_targets = np.concatenate([valid_targets,np.array([ref_ph])])
            probs = counts_all / counts_sum_all
        replacement_table[ref_ph] = (valid_targets, probs)
    print(replacement_table)
    print(total_correct/counter)
    print(err_count/counter)
    return replacement_table

conf_df = pd.read_csv('phoneme_confusion_matrix.csv',index_col=0)
replacement_table = build_replacement_table(conf_df, LABELS)
# print(replacement_table)

NOISE_PHONEMES = [p for p in LOGIT_TO_PHONEME if p not in SPECIAL_TOKENS]
INSERT_PHONEMES = ['AA','AH','HH','K','L','M','N','R']
def noise_phoneme_tokens(
    tokens,
    insert_prob: float = 0.010,
    replacement_table=replacement_table,
):
    
    # Maybe keep the whole sequence untouched
    if random.random() < 0.1:
        return tokens
    new_tokens = []
    for t in tokens:
        ts = t.strip()
        # Do not corrupt separators or special tokens
        if ts in SPECIAL_TOKENS:
            new_tokens.append(t)
            continue

        # Switch (substitute)
        if replacement_table is not None:
            targets, probs = replacement_table[ts]
            # pick according to confusion-based probs
            new_t = random.choices(targets, weights=probs, k=1)[0]
        if new_t=='<DEL>': continue
        new_tokens.append(new_t)

    if random.random() < insert_prob:
        if INSERT_PHONEMES:
            insert_ph = random.choice(INSERT_PHONEMES)
            pos = random.randint(0, len(new_tokens))
            new_tokens.insert(pos, insert_ph)
    return new_tokens

# def noise_phoneme_tokens(
#     tokens,
#     keep_clean_prob: float = 0.10,
#     delete_prob: float = 0.025,
#     switch_prob: float = 0.07,
#     insert_prob: float = 0.020,
# ):
#     """
#     tokens: list of strings, like ["B", "R", "IH", "NG", "|", "IH", "T", "|", ...]
#     There is a 25 percent chance to keep the whole sample clean.
#     Otherwise:
#       per token: 10 percent delete, 10 percent switch, rest keep
#       after that: 10 percent chance to insert one extra phoneme at a random position.
#     """
#     # 25 percent chance to keep as is
#     if random.random() < keep_clean_prob:
#         return tokens

#     # 1. delete and switch
#     new_tokens = []
#     for t in tokens:
#         t_stripped = t.strip()
#         # keep separators untouched
# #         if t_stripped == "|":
# #             new_tokens.append(t)
# #             continue

#         # delete this phoneme
#         if random.random() < delete_prob:
#             continue

#         # switch to a different phoneme
#         elif random.random() < switch_prob:
#             new_t = t
#             if len(NOISE_PHONEMES) > 1:
#                 while new_t == t:
#                     new_t = random.choice(NOISE_PHONEMES)
#             else:
#                 new_t = NOISE_PHONEMES[0]
#             new_tokens.append(new_t)
#         else:
#             new_tokens.append(t)

#     # 2. optional single insertion at random position
#     if random.random() < insert_prob and NOISE_PHONEMES:
#         insert_ph = random.choice(INSERT_PHONEMES)
#         if insert_ph!='<DEL>': 
#             pos = random.randint(0, len(new_tokens))
#             new_tokens.insert(pos, insert_ph)
#     return new_tokens


def noise_phoneme_line_in_text(text: str) -> str:
    # group(1): "Phonemes: "
    # group(2): the phoneme sequence only
    # group(3): the user-side "<|im_end|>" and any whitespace before it
    pattern = r"(Phonemes:\s*)(.+)"
    m = re.search(pattern, text, flags=re.DOTALL)
    if not m:
        return text  # no phoneme line found, do nothing

    prefix = m.group(1)       # "Phonemes: "
    phoneme_str = m.group(2)  # "B R IH NG | IH T | K L OW S ER |"
#     suffix = m.group(3)       # " <|im_end|>"

    tokens = phoneme_str.split()
    noisy_tokens = noise_phoneme_tokens(tokens)
    noisy_phoneme_str = " ".join(noisy_tokens)
    noisy_text = text[:m.start()] + prefix + noisy_phoneme_str + text[m.end():]
    return noisy_text

from dataclasses import dataclass
from typing import List, Dict, Any
import torch
from transformers import PreTrainedTokenizerBase

@dataclass
class OnlinePhonemeNoiseCollator:
    tokenizer: PreTrainedTokenizerBase
    max_length: int = 300
    apply_noise: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        texts = []
        for f in features:
            messages = f["messages"]

            # Optionally apply phoneme noise only in user message
            if self.apply_noise:
                new_messages = []
                for msg in messages:
                    msg = dict(msg)  # shallow copy
                    if msg.get("role") == "user" and "Phonemes:" in msg.get("content", ""):
                        msg["content"] = noise_phoneme_line_in_text(msg["content"])
                    new_messages.append(msg)
                messages = new_messages
                
            # Build chat text using Qwen template
            text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
        
        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        batch["labels"] = batch["input_ids"].clone()
        return batch

{'AA': (array(['AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER',
       'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG',
       'OW', 'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W',
       'Y', 'Z', 'ZH', '<DEL>', 'AA'], dtype='<U5'), array([0.        , 0.01818182, 0.        , 0.        , 0.        ,
       0.01818182, 0.        , 0.        , 0.        , 0.01818182,
       0.01818182, 0.        , 0.        , 0.        , 0.        ,
       0.01818182, 0.        , 0.        , 0.        , 0.01818182,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.01818182,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.03636364, 0.83636364])), 'AE': (array(['AA', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER',
       'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG',
       'OW', 'OY', 'P', 'R', 'S'

#### Set train config

In [16]:
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
import torch
MODEL_SIZE = '7B'
model_size = '7b'
# -------------------------
# 1. Load tokenizer & base model name
# -------------------------
model_name = f".\models--Qwen--Qwen2.5-{MODEL_SIZE}-Instruct\snapshots\model"
model_path = f".\Qwen2.5-{MODEL_SIZE}-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Qwen sometimes has no pad_token set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# 2. Load JSONL and split into train / val (72 samples)
# -------------------------
# ds = load_dataset(
#     "json",
#     data_files={"train": "./data/phoneme_train.jsonl"},
# )

# ds2 = load_dataset(
#     "json",
#     data_files={"train": "./data/phoneme_sb.jsonl"},
# )

# all_idx = len(ds['train'])

# tran_idx = all_index[:all_idx//10*9+9]
# test_idx = all_index[all_idx//10*9+9:]

# full = ds["train"]
# print("Total samples:", len(full))

# train_ds = full.select(tran_idx)
# val_ds  = full.select(test_idx)

# train_ds = concatenate_datasets([train_ds, ds2["train"]])
# print("Train samples:", len(train_ds))
# print("Val samples:", len(val_ds))


# -------------------------
# 3. Build conversation messages from phonemes + target
# -------------------------
def make_conversation(example):
    phoneme_str = example["phonemes"]
    target = example["target"]

    system = (
        "Convert phonemes into English."
    )
    user = (
        "Convert this phoneme sequence into a fluent English sentence.\n"
        f"Phonemes: {phoneme_str}"
    )
    assistant = target

    return {
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]
    }


def build_messages(example):
    conv = make_conversation(example)
    example["messages"] = conv["messages"]
    return example

# train_ds = train_ds.map(build_messages)
# val_ds = val_ds.map(build_messages)

Phoneme2Sentence = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

Phoneme2Sentence.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # typical for Qwen
    lora_dropout=0.01,
    bias="none",
    task_type="CAUSAL_LM",
)

Phoneme2Sentence = get_peft_model(Phoneme2Sentence, lora_config)
Phoneme2Sentence.print_trainable_parameters()

major, _ = torch.cuda.get_device_capability()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.0662


In [17]:
# training_args = TrainingArguments(
#     output_dir=f"./qwen25_{model_size}_phoneme_lora_r16",
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,
#     num_train_epochs=2,
#     learning_rate=4e-5,
#     warmup_ratio=0.001,
#     evaluation_strategy="steps",
#     eval_steps=100,
#     save_steps=100,
#     logging_steps=100,
#     bf16=torch.cuda.is_available(),  # or fp16 if your GPU prefers
#     save_total_limit=2,
#     report_to="none",
#     remove_unused_columns=False,
# )

# data_collator = OnlinePhonemeNoiseCollator(
#     tokenizer=tokenizer,
#     max_length=300,
#     apply_noise=True,
# )

# trainer = Trainer(
#     model=Phoneme2Sentence,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,  
#     data_collator=data_collator,
# )

# if False:
#     trainer.train()
#     Phoneme2Sentence.save_pretrained(f"qwen25_{model_size}_phoneme_lora")
#     tokenizer.save_pretrained(f"qwen25_{model_size}_phoneme_lora")

In [18]:
from peft import PeftModel

import json
import shutil

cfg_path = r"qwen25_7b_phoneme_lora\checkpoint-final\adapter_config.json"

# Keep an original backup
shutil.copy(cfg_path, cfg_path + ".original_backup")

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

remove_fields = [
    "alora_invocation_tokens",
    "arrow_config",
    "corda_config",
    "ensure_weight_tying",
    "eva_config",
    "exclude_modules",
    "lora_bias",
    "peft_version",
    "qalora_group_size",
    "target_parameters",
    "trainable_token_indices",
    "use_qalora",
]

for key in remove_fields:
    cfg.pop(key, None)

with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("Compatibility config written.")


base = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map=None,     
    low_cpu_mem_usage=False    
)
base = base.to('cuda')
Phoneme2Sentence = PeftModel.from_pretrained(base, f"qwen25_{model_size}_phoneme_lora/checkpoint-final")
Phoneme2Sentence = Phoneme2Sentence.to('cuda')
Phoneme2Sentence.eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)

Compatibility config written.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
from typing import List, Tuple

# 1) Inventory adapter and CTC-style collapse
CANONICAL_MAP = {
'BLANK',    # "BLANK" = CTC blank symbol
'AA', 'AE', 'AH', 'AO', 'AW',
'AY', 'B', 'CH', 'D', 'DH',
'EH', 'ER', 'EY', 'F', 'G',
'HH', 'IH', 'IY', 'JH', 'K',
'L', 'M', 'N', 'NG', 'OW',
'OY', 'P', 'R', 'S', 'SH',
'T', 'TH', 'UH', 'UW', 'V',
'W', 'Y', 'Z', 'ZH',
'|',    # "|" = silence token
}
DEFAULT_BLANKS = ("BLANK")

def normalize_symbols(seq: List[str]) -> List[str]:
    out = []
    for t in seq:
        t_up = t.upper()
        out.append(CANONICAL_MAP.get(t_up, t_up))
    return out

def collapse_ctc(tokens: List[str], blank_tokens: Tuple[str, ...] = DEFAULT_BLANKS) -> List[str]:
    out, prev = [], None
    for p in tokens:
        if p in blank_tokens:
            continue 
        if p != prev:
            out.append(p)
        prev = p
    return out

# 2) Prompt builder
def _build_messages(phoneme_str: str) -> list:
    sys_msg = {
        "role":"system",
        "content":(
            "Convert phonemes into English."
        )
    }
    
    task = {"role":"user","content":(
            "Convert this phoneme sequence into a fluent English sentence.\n"
            f"Phonemes: {phoneme_str}"
    )}
    return [sys_msg] + [task]

# 3) Main decode function (call this)
def decode_with_llm(
    phoneme_tokens: List[str],
    model,
    temperature: float = 0,
    top_p: float = 1,
    max_new_tokens: int = 64,
    repetition_penalty: float = 1.00,
) -> str:
    """
        Input: list of phoneme tokens, e.g. ["DH","IH","S","IH","Z","AH","T","EH","S","T"]
        Output: a fluent English sentence (str).
    """


    # If you want CTC collapse, you can restore this:
#     collapsed = collapse_ctc(normalize_symbols(phoneme_tokens))
    # if not collapsed:
    #     return ""
    # phoneme_str = " ".join(collapsed)

    phoneme_str = " ".join(phoneme_tokens)
    messages = _build_messages(phoneme_str)



    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            eos_token_id=tokenizer.eos_token_id,
        )

    # *** key change: remove the prompt part ***
    input_len = inputs["input_ids"].shape[1]
    generated_ids = out_ids[0]#[input_len:]

    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # keep only the final line in case the model prints extra stuff
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[-1] if lines else text


In [20]:
def _parse_session_date(session_name: str):
    # session folder like 't15.2023.08.13' -> (2023, 8, 13)
    m = re.search(r'(\d{4})\.(\d{2})\.(\d{2})', session_name)
    return tuple(map(int, m.groups())) if m else (9999, 99, 99)

def _discover_test_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_test.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _discover_val_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_val.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _sorted_trials(hf):
    trials = [k for k in hf.keys() if k.startswith("trial_")]
    def idx(k):
        m = re.search(r'trial_(\d+)', k)
        return int(m.group(1)) if m else 0
    return sorted(trials, key=idx)

def indices_to_phonemes(tensor_indices, remove_blank=True):
    # Convert tensor to list if needed
    if isinstance(tensor_indices, torch.Tensor):
        tensor_indices = tensor_indices.tolist()
    
    phonemes = [LOGIT_TO_PHONEME[i] for i in tensor_indices]
    
    if remove_blank:
        phonemes = [p for p in phonemes if p not in ['BLANK',]]
    return phonemes

INPUT_DIR = Path('./t15_copyTask_neuralData/hdf5_data_final')
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# importlib.reload(lmdecode)
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct" #"F:\\Kaggle\\models--Qwen--Qwen2-7B-Instruct\\snapshots\\model"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# LMMmodel = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, #mistralai/Mistral-7B-Instruct-v0.3,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto",
#     trust_remote_code=True,
# )

results = []
model.eval()
test_files = _discover_test_files_in_order(INPUT_DIR)
val_files = _discover_val_files_in_order(INPUT_DIR)

limit, count = 5000, 0
results, pred_ph = [], []
targets, target_ph = [], []
for test_path in tqdm(val_files, desc="Inference with LM (chronological)"):
    with h5py.File(test_path, "r") as hf:
        match = re.search(r"\d{4}\.\d{2}\.\d{2}", str(test_path))
        date = torch.tensor([dates_dict[match.group(0)]]).to(device)
        for trial_name in _sorted_trials(hf):
            feats = hf[trial_name]["input_features"][()].astype("float32")
            transcript = hf[trial_name]["transcription"][()]
            transcript = codes_to_sentence(transcript)
            tgt_phoneme = hf[trial_name]['seq_class_ids'][()].astype('int64')
            tgt_phoneme = tgt_phoneme[tgt_phoneme!=0]
            length = feats.shape[0]
            while length % downsample_factor != 0:
                length += 1
            pad = length - feats.shape[0]
            if pad > 0:
                feats = np.concatenate([feats, np.zeros((pad, feats.shape[1]), dtype=np.float32)], axis=0)
#                 feats = np.concatenate([np.zeros((25, feats.shape[1]), dtype=np.float32) ,feats], axis=0)
            feats = torch.tensor(feats).to(device)[None,:,:].transpose(2,1)
            feats = feats[:,256:,:]
            feats = gauss_smooth(feats.transpose(2,1),device)
            ordered_texts = model(feats,date).argmax(-1)
            ids = torch.tensor([p for j,p in enumerate(ordered_texts[0]) if (j==0 or p!=ordered_texts[0][j-1]) and p!=0])
            toks = indices_to_phonemes(ids)
            tgt_ph = indices_to_phonemes(tgt_phoneme)
#             chunks = lmdecode.split_by_bar(toks, bar="|")
#             sent = lmdecode.decode_chunks_with_word_lm(chunks,lexicon_path="data/lexicon.txt",word_lm_bin="data/word_o4.bin",max_edits=2,
#                     edit_penalty=2, word_bonus=2.0,beam_size=60)
#             sent = " ".join([w for w in sent[0].split() if w != "<unk>"])
#             sent = decode_with_llm(
#                 toks,
#                 Phoneme2Sentence, 
#                 temperature=0,
#                 top_p=1,
#                 max_new_tokens=64,
#                 repetition_penalty=1.05,
#             )
#             print(sent)
            pred_ph.append(toks)
            target_ph.append(tgt_ph)
#             results.append(sent)
#             targets.append(transcript)
#             print('target:',indices_to_phonemes(tgt_pheneme), '\npredicts:',toks)
#             print('target:',transcript, '\npredicts:',f"{sent}")
            count+=1
            if count==limit:
                break
    if count==limit:
        break
            
# final_results = pd.DataFrame({
#     "id": list(range(len(results))),
#     "text": results
# })
# final_targets = pd.DataFrame({
#     "id": list(range(len(results))),
#     "text": targets
# })

wers = []
preds, preds_ph,tgts, tgts_ph = [] , [], [], []
# for p, r in zip(results, targets):
#     preds.append(" ".join(map(str, p)))
#     tgts.append(" ".join(map(str, r)))

for p, r in zip(pred_ph, target_ph):
    preds_ph.append(" ".join(map(str, p)))
    tgts_ph.append(" ".join(map(str, r)))

w = jiwer.wer(preds, tgts)
print(w)
# Print average WER
# print("Average WER:", w)
# w1 = jiwer.wer(preds_ph, tgts_ph)


Inference with LM (chronological):   0%|                                                        | 0/41 [00:00<?, ?it/s]e:\python\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
Inference with LM (chronological): 100%|███████████████████████████████████████████████| 41/41 [00:52<00:00,  1.29s/it]

0


In [35]:
def _parse_session_date(session_name: str):
    # session folder like 't15.2023.08.13' -> (2023, 8, 13)
    m = re.search(r'(\d{4})\.(\d{2})\.(\d{2})', session_name)
    return tuple(map(int, m.groups())) if m else (9999, 99, 99)

def _discover_test_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_test.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _discover_val_files_in_order(input_dir: Path):
    # Find all data_test.hdf5 and sort by session date ascending
    items = []
    for p in input_dir.rglob("data_val.hdf5"):
        session = p.parent.name  # e.g., 't15.2023.08.13'
        items.append((p, session))
    items.sort(key=lambda x: _parse_session_date(x[1]))
    return [p for p, _ in items]

def _sorted_trials(hf):
    trials = [k for k in hf.keys() if k.startswith("trial_")]
    def idx(k):
        m = re.search(r'trial_(\d+)', k)
        return int(m.group(1)) if m else 0
    return sorted(trials, key=idx)

def indices_to_phonemes(tensor_indices, remove_blank=True):
    """
    Convert a tensor of indices to a list of phoneme tokens.
    Args:
        tensor_indices: torch.Tensor or list[int]
        remove_blank: whether to skip BLANK, <pad>, SIL tokens
    """
    # Convert tensor to list if needed
    if isinstance(tensor_indices, torch.Tensor):
        tensor_indices = tensor_indices.tolist()
    
    phonemes = [LOGIT_TO_PHONEME[i] for i in tensor_indices]
    
    if remove_blank:
        phonemes = [p for p in phonemes if p not in ['BLANK']]
    return phonemes

INPUT_DIR = Path('./t15_copyTask_neuralData/hdf5_data_final')
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct" #"F:\\Kaggle\\models--Qwen--Qwen2-7B-Instruct\\snapshots\\model"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# LMMmodel = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, #mistralai/Mistral-7B-Instruct-v0.3,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto",
#     trust_remote_code=True,
# )

results = []
model.eval()
test_files = _discover_test_files_in_order(INPUT_DIR)
val_files = _discover_val_files_in_order(INPUT_DIR)

limit, count = 20, 0
results = []
targets = []
for test_path in tqdm(test_files, desc="Inference with LLM"):
    with h5py.File(test_path, "r") as hf:
        match = re.search(r"\d{4}\.\d{2}\.\d{2}", str(test_path))
        date = [dates_dict[match.group(0)]]
        for trial_name in _sorted_trials(hf):
#             with h5py.File(test_path, "r") as hf:
#                 print("Top-level keys:")
#                 print(list(hf[trial_name].keys()))
            feats = hf[trial_name]["input_features"][()].astype("float32")
#             transcript = hf[trial_name]["transcription"][()]
#             transcript = codes_to_sentence(transcript)
            length = feats.shape[0]
            while length % downsample_factor != 0:
                length += 1
            pad = length - feats.shape[0]
            if pad > 0:
                feats = np.concatenate([feats, np.zeros((pad+25, feats.shape[1]), dtype=np.float32)], axis=0)
                feats = np.concatenate([np.zeros((25, feats.shape[1]), dtype=np.float32) ,feats], axis=0)
                
            feats = torch.tensor(feats).to(device)[None,:,:]
            feats = gauss_smooth(feats,device)
            feats = feats[:,256:,:]
            ordered_texts = model(feats,date).argmax(-1)
            ids = torch.tensor([p for j,p in enumerate(ordered_texts[0]) if (j==0 or p!=ordered_texts[0][j-1]) and p!=0])
            toks = indices_to_phonemes(ids)
            print("predicted phonemes:",toks)
            sent = decode_with_llm(
                toks,
                Phoneme2Sentence,
                temperature=0,
                top_p=1,
                max_new_tokens=32,
                repetition_penalty=1.05,
            )
            print("predicted sentence:",sent)
#             print("target phonemes:",transcript)
            print()
#             print('target:',indices_to_phonemes(tgt_pheneme), '\npredicts:',toks)
#             print('target:',transcript, '\npredicts:',f"{sent}")
            count+=1
    if count>=limit:
        break
        

Inference with LLM:   0%|                                                                       | 0/41 [00:00<?, ?it/s]

predicted phonemes: ['AY', '|', 'G', 'EH', 'T', '|', 'T', 'AY', 'ER', 'D', '|', 'W', 'IH', 'DH', '|', 'DH', 'AH', '|', 'S', 'K', 'AO', 'NG', '|', 'AH', 'N', 'D', '|', 'D', 'EY', 'S', '|', 'R', 'IY', 'T', 'IY', 'N', '|']
predicted sentence: i get tired with the song and dance routine

predicted phonemes: ['IH', 'M', 'AO', 'R', 'Z', 'AH', 'S', 'IH', '|', 'K', 'IY', 'R', '|']
predicted sentence: impossibly clear

predicted phonemes: ['Y', 'UW', '|', 'R', 'IY', 'EY', 'T', '|', 'AH', '|', 'B', 'EY', 'G', 'ER', '|', 'S', 'ER', 'P', 'R', 'AY', 'S', '|']
predicted sentence: you create a bigger surprise

predicted phonemes: ['AY', '|', 'TH', 'IH', 'NG', 'K', '|', 'M', 'EY', 'V', 'IY', '|', 'Y', 'UW', '|', 'L', 'UH', 'K', '|', 'AE', 'T', '|', 'IH', 'T', '|']
predicted sentence: i think maybe you look at it

predicted phonemes: ['SH', 'OW', '|', 'DH', 'AE', 'T', '|', 'DH', 'EY', '|', 'D', 'UW', '|', 'HH', 'AE', 'V', '|', 'P', 'R', 'AA', 'B', 'L', 'AH', 'M', 'Z', '|']
predicted sentence: show that

Inference with LLM:   0%|                                                                       | 0/41 [00:35<?, ?it/s]

predicted sentence: that kind you've read recently

